# CYGNSS L1 AZ: TC and IV maps (SMAP L3 + ASCAT H121)

Per-tile triple collocation and independent-validation maps for the fixed-operator arms, 2020–2022.
The plotting code lives in `scripts/iv_tc_az_maps.py`; this notebook only sets options and calls it, so the same figures can be made from the command line:

`python -I scripts/iv_tc_az_maps.py --root <iv_tc_output> --out <figdir>`

**Conventions (following the CYGNSS L3 DA paper):**
- Every tile keeps its own value. The maps are drawn on one fixed common mask, the L1 domain (`in_l1_mask`, ≥100 L1 obs).
- Grey tiles are inside the mask but fail the validity rule (too few samples, non-finite, or a missing run). Very light tiles are outside the mask.
- TC: ΔR²_mod and % Δσ²_ε are drawn so that **red/positive = arm better than the OL** (R² clamped to (0,1], tiny baseline σ² masked before the percentage).
- IV: ΔanomR = anomaly correlation of the arm minus the OL, against each reference separately; red/positive = better.
- Domain means are summaries of these per-tile fields and always report how many tiles contributed.
- DA_SMAP is not included: it assimilates SMAP Tb, so SMAP L3 is not an independent reference for it.

In [ ]:
import sys
from pathlib import Path

SCRIPTS = Path.cwd().resolve().parent / "scripts"          # run from notebooks/
if not SCRIPTS.exists():
    SCRIPTS = Path("projects/CYGNSS_L1_AZ/scripts").resolve()
sys.path.insert(0, str(SCRIPTS))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import iv_tc_az_maps as m

%matplotlib inline

## Options

In [ ]:
ROOT   = Path("/discover/nobackup/projects/land_da/cygl1_operator_test/iv_tc_output")   # step4_tc / step4_iv live here
OUTDIR = None            # e.g. Path("<land_da project path>/iv_tc_figs") to also save PNGs; None = display only
START, END = "20200101", "20221231"
TC_NMIN = 20             # TC min sample count per tile (must match the files that were computed)
IV_NMIN = 100            # IV min sample count per tile
REF = "OL"
ARMS = ["L1_filt275", "L3"]          # arm runs to compare with the OL (run labels, see m.RUNS)
USE_MASK = True          # restrict to the L1 domain; False shows all 909 tiles

## Load

The tile map joins the TC/IV cell index (`idx0`) to the domain tiles and carries the L1 mask. It is rebuilt by `scripts/build_iv_tc_tile_map.py` if missing.

In [ ]:
tm = m.load_tile_map()
print(f"{len(tm)} domain tiles, {int(tm.in_l1_mask.sum())} in the L1 mask")
runs = [REF] + ARMS
tc, iv = m.load_all(ROOT, tm, runs, START, END, TC_NMIN, IV_NMIN)
print("TC triplet:", tc[REF].attrs.get("names"))
pd.DataFrame({r: {"tiles with TC": int(tc[r].r2_mod.notna().sum()),
                  f"TC N>={TC_NMIN}": int((tc[r].N >= TC_NMIN).sum())} for r in runs})

## Reference run: where is the model weak?

OL model error variance σ²_ε, R²_mod, and the number of days behind each TC estimate.

In [ ]:
fig = m.fig_tc_baseline(tc, tm, REF, savepath=(OUTDIR / "tc_baseline_OL.png") if OUTDIR else None)
plt.show()

## TC: arm − OL

Left: ΔR²_mod. Right: % reduction in σ²_ε. Positive (red) = arm better. The footer gives the number of valid tiles and the fraction with improved R².

In [ ]:
cmps = {}
for arm in ARMS:
    cmps[arm] = m.compare_tc(tc[REF], tc[arm], tm, nmin=TC_NMIN, use_mask=USE_MASK)
    fig = m.fig_tc_delta(cmps[arm], m.RUNS.get(arm, arm), REF,
                         savepath=(OUTDIR / f"tc_delta_{arm}.png") if OUTDIR else None)
    plt.show()

## IV: arm − OL anomaly correlation

ΔanomR against SMAP L3 and against ASCAT H121 separately, the sign-agreement class map (red = both references say the arm is better), and the mean of the two on tiles where both exist.

In [ ]:
drs = {}
for arm in ARMS:
    fig, drs[arm] = m.fig_iv_delta(iv, tm, arm, REF, m.RUNS.get(arm, arm),
                                   savepath=(OUTDIR / f"iv_delta_{arm}.png") if OUTDIR else None)
    plt.show()

## Domain summaries (after the maps)

Per-arm means over the common tile set with the tile counts behind them.

In [ ]:
tab = m.summary_table(tc, iv, tm, runs, ref=REF, tc_nmin=TC_NMIN, iv_nmin=IV_NMIN)
tab.round(4).T

## Per-tile table

One row per domain tile with the TC and IV deltas, for any further slicing (L1 obs count, location, ...).

In [ ]:
per_tile = tm[["tile_axis_index", "tile_id", "idx0", "com_lon", "com_lat", "n_l1_obs", "in_l1_mask"]].copy()
for arm in ARMS:
    per_tile[f"dR2_mod_{arm}"] = cmps[arm].d_r2
    per_tile[f"pct_sig2_better_{arm}"] = cmps[arm].pct_sig2_inv
    for s in ("smap_l3", "ascat_h121"):
        per_tile[f"dAnomR_{s}_{arm}"] = m.compare_iv(iv[(s, REF)], iv[(s, arm)], tm, nmin=IV_NMIN, use_mask=USE_MASK)[1]
per_tile.describe().T[["count", "mean", "std", "min", "max"]].round(4)
# per_tile.to_csv(OUTDIR / "iv_tc_per_tile.csv", index=False)   # if OUTDIR is set